In [2]:
import pyproj
import math
import numpy as np
import matplotlib.pyplot as plt

In [3]:
def midpoint(A, B, t=0.5):
    deltaX = -A[0] + B[0]
    deltaY = -A[1] + B[1]
    deltaZ = -A[2] + B[2]

    return A[0] + deltaX * t, A[1] + deltaY * t, A[2] + deltaZ * t

In [4]:
latA, lonA, altA_m = (40.865339, -74.030096, 100.0)
latB, lonB, altB_m = (40.682415, -73.871022, 100.0)
midpoint((latA, lonA, altA_m), (latB, lonB, altB_m))

(40.773877, -73.950559, 100.0)

In [22]:
nys_crs = pyproj.CRS.from_string("EPSG:6539+6360")
gps_crs = pyproj.CRS.from_string("EPSG:4326+5773")
ecef_cr = pyproj.CRS.from_string("EPSG:4978")

gps_to_ecef = pyproj.Transformer.from_crs(gps_crs, ecef_cr, always_xy=False)
gps_to_nys = pyproj.Transformer.from_crs(gps_crs, nys_crs, always_xy=False)
ecef_to_nys = pyproj.Transformer.from_crs(ecef_cr, nys_crs, always_xy=False)

xA, yA, zA = gps_to_nys.transform(latA, lonA, altA_m)
xB, yB, zB = gps_to_nys.transform(latB, lonB, altB_m)

xGA, yGA, zGA = gps_to_ecef.transform(latA, lonA, altA_m)
xGB, yGB, zGB = gps_to_ecef.transform(latB, lonB, altB_m)

deltaX = xB - xA
deltaY = yB - yA
deltaZ = zB - zA

t = 0.5
xC, yC, zC = midpoint((xA, yA, zA), (xB, yB, zB), t)
xGC, yGC, zGC = midpoint((xGA, yGA, zGA), (xGB, yGB, zGB), t)

xC_2, yC_2, zC_2 = ecef_to_nys.transform(xGC, yGC, zGC)
(xC, yC, zC), (xC_2, yC_2, zC_2), zC - zC_2

((997974.0998292989, 221235.17465089564, 328.08333333333337),
 (997974.0532765132, 221235.194446586, 289.90646187919896),
 38.176871454134414)

In [23]:
earth_radius_m = 6_369_160 # (approx) In NYC

L_m = np.linalg.norm(np.array([xGA-xGB, yGA-yGB, zGA-zGB]))
error_m = math.sqrt(earth_radius_m**2 - (L_m / 2 - L_m*t)**2) - math.sqrt(earth_radius_m**2 - (1/4) * L_m**2)
error_m / 0.30480061

38.18327958427803

In [24]:
L_m*t

np.float64(12175.881382686282)